### Fine-tuning a BERT model for aspect sentiment classification (ASC)

This notebook finetunes and evaluates a BERT model on ASC by utilising pairwise classification where each pair consists of a (comment, aspect) and has a corresponding sentiment label. 
Label = 2 is positive, label = 1 is neutral and label = 0 is negative. 

- 3 datasets were used for fine-tuning and evaluation. These datasets can be found in the dataset folder within this directory.
- for each dataset, this fine-tuning and evaluation pipeline was run 3 times using a different train-test split each time, namely with seed 42, 55 and 777.
- The average score of each metric across the 3 runs was computed and the results are listed in the final report

To run, activate the `bert_environment` conda environment specified in the environements directory. Furthermore, replace the API_KEY placeholder with a real OpenAI API key.

In [1]:
import os
import numpy as np
import torch
import pandas as pd
from torch.utils.data import Dataset
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import (BertTokenizerFast, BertForSequenceClassification, Trainer,TrainingArguments,)
from datasets import Dataset as HFDataset

# Loading dataset to be used for model fine-tuning and evaluation
input_path = "dataset/edurabsa_dataset_2.xlsx"
data = pd.read_excel(input_path, engine="openpyxl")

# enabling mps backend
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
device = "mps"

/opt/anaconda3/envs/bert_absa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# balancing class distribution in dataset

# split into train and test datasets
hf_dataset = HFDataset.from_pandas(data)
train_test_split = hf_dataset.train_test_split(test_size=0.2, seed=42) 
train_data = train_test_split["train"].to_pandas()
test_data  = train_test_split["test"].to_pandas()

# upsample records from neutral class to balance training dataset
neg = train_data[train_data["label"] == 0]
neu = train_data[train_data["label"] == 1]
pos = train_data[train_data["label"] == 2]
print("Original class distribution")
print(f"negative: {len(neg)}")
print(f"neutral: {len(neu)}")
print(f"positive: {len(pos)}")

factor = max(len(neg), len(pos))// max(1, len(neu))
neu_upsampled = pd.concat([neu]*factor, ignore_index=True)
balanced_train_data = pd.concat([neg, pos, neu_upsampled], ignore_index=True)
balanced_train_data = balanced_train_data.sample(frac=1, random_state=42)

neg = balanced_train_data[balanced_train_data["label"] == 0]
neu = balanced_train_data[balanced_train_data["label"] == 1]
pos = balanced_train_data[balanced_train_data["label"] == 2]
print("New class distribution")
print(f"negative: {len(neg)}")
print(f"neutral: {len(neu)}")
print(f"positive: {len(pos)}")

Original class distribution
negative: 558
neutral: 182
positive: 868
New class distribution
negative: 558
neutral: 728
positive: 868


In [3]:
# initialise tokenizer and model
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3).to(device)
max_len=128

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# convert data into a form compatible for fine-tuning with BERT
class ABSADataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.data = data.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        sentence = row["sentence"]
        aspect   = row["aspect"]
        label    = int(row["label"])

        inputs = self.tokenizer(
            sentence, aspect,
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item
    
train_dataset = ABSADataset(balanced_train_data, tokenizer, max_length=max_len)
test_dataset  = ABSADataset(test_data,  tokenizer, max_length=max_len)

In [5]:
# defining evaluation metrics with macro averaging
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

In [6]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,               
    per_device_train_batch_size=8,     
    per_device_eval_batch_size=8,
    warmup_steps=0,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",
    use_mps_device=True if device == "mps" else False, 
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()
print(trainer.evaluate())

/opt/anaconda3/envs/bert_absa/lib/python3.10/site-packages/transformers/training_args.py:2278: UserWarning: `use_mps_device` is deprecated and will be removed in version 5.0 of 🤗 Transformers. `mps` device will be used by default if available similar to the way `cuda` device is used.Therefore, no action from user is required. 
  warnings.warn(
/opt/anaconda3/envs/bert_absa/lib/python3.10/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,1.103100
100,0.913800
150,0.933900
200,0.832300
250,0.792000
300,0.649800
350,0.537000
400,0.585200
450,0.530500
500,0.422700


/opt/anaconda3/envs/bert_absa/lib/python3.10/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/envs/bert_absa/lib/python3.10/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 1.0079799890518188, 'eval_accuracy': 0.8039702233250621, 'eval_f1': 0.66284624180517, 'eval_precision': 0.6674792054226271, 'eval_recall': 0.6595988775173325, 'eval_runtime': 3.9637, 'eval_samples_per_second': 101.673, 'eval_steps_per_second': 12.867, 'epoch': 4.0}
